# 🌊 Aether Flood Prediction — v3 RANK-1 ATTACK

## Leaderboard situation
| # | Team | KGE |
|---|---|---|
| 1 | AI Sniffers | 0.99963 |
| 2 | siddhxsh | 0.99962 |
| 3 | Syed Mustafa Hussain | 0.99863 |
| **4** | **Us (Manas singh)** | **0.99754** |

Gap to close: **0.00209**

## What v3 adds over v2 (scored 0.99754)
| Addition | Expected KGE gain |
|---|---|
| Lags 7, 14, 21 days — river has long memory | +0.001 |
| Rolling stats at 3d, 14d, 30d windows | +0.0005 |
| Flow × rain interaction (flood onset signal) | +0.0003 |
| Monthly percentile rank (seasonal context) | +0.0003 |
| Post-peak decay flag | +0.0002 |
| Ridge meta-learner stacking (XGB+LGB+CAT → Ridge) | +0.0005 |
| Finer blend on log scale with scipy minimize | +0.0001 |

**Target leaderboard KGE: 0.9996+**

## 0 · Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from scipy.optimize import minimize
from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor

RANDOM_STATE = 42
print('All imports OK')

## 1 · Load Data

In [ ]:
BASE = r'C:\Users\user\Desktop\Aether comp\aether-iiit-lucknow-stream-flow-prediction-challenge-2026'

train = pd.read_csv(f'{BASE}/train_flood.csv')
test  = pd.read_csv(f'{BASE}/test_flood.csv')
sub   = pd.read_csv(f'{BASE}/sample_submission.csv')

print(f'Train : {train.shape}')
print(f'Test  : {test.shape}')
print(f'Sub   : {sub.shape}')
train.head(3)

## 2 · Metrics

In [ ]:
def compute_kge(pred, actual):
    r     = np.corrcoef(actual, pred)[0, 1]
    alpha = pred.std()  / (actual.std()  + 1e-8)
    beta  = pred.mean() / (actual.mean() + 1e-8)
    return 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)

def compute_nse(pred, actual):
    return 1.0 - np.sum((actual - pred)**2) / (np.sum((actual - actual.mean())**2) + 1e-8)

def regime_analysis(pred, actual, label='Model'):
    p50 = np.percentile(actual, 50)
    p95 = np.percentile(actual, 95)
    masks = {
        'Low flow   (< p50) ': actual <= p50,
        'Rising     (p50-95)': (actual > p50) & (actual <= p95),
        'Flood peak (> p95) ': actual > p95,
    }
    print(f'\n── {label} Regime Analysis ──')
    for name, mask in masks.items():
        print(f'  {name}  n={mask.sum():>8,}  KGE={compute_kge(pred[mask],actual[mask]):.4f}  NSE={compute_nse(pred[mask],actual[mask]):.4f}')
    print(f'  Overall               n={len(actual):>8,}  KGE={compute_kge(pred,actual):.4f}  NSE={compute_nse(pred,actual):.4f}')

print('Metrics defined.')

## 3 · Feature Engineering v3 (RANK-1 edition)

In [ ]:
class FloodFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    v3 adds over v2:
      flow_lag7/14/21       : long-memory lags (rivers remember weeks of rain)
      rolling_3d stats      : fast-response window (3-day mean, std, max)
      rolling_14d/30d stats : slow-memory windows
      flow_x_rain3d         : flow * 3d rain interaction (flood onset signal)
      flow_x_rain7d         : flow * 7d rain interaction
      monthly_pct_rank      : where today's flow sits vs same month historically
      post_peak_decay       : flag for flow falling after a recent peak
      log_flow_lag7/14/21   : log of long lags
    """

    RAIN_COLS = [
        'antecedent_rain_3d_sum', 'antecedent_rain_7d_sum',
        'antecedent_rain_15d_sum', 'antecedent_rain_30d_sum', 'antecedent_rain_60d',
    ]
    EXTRA_LOG_COLS = [
        'soil_saturation_score', 'monsoon_cumulative_rain',
        'rain_soilmoisture_interaction', 'uparea_rain_interaction', 'rain_basinsize_interaction',
    ]

    def fit(self, X, y=None):
        self.train_tail_ = (
            X[['streamflow_today_cumecs', 'flow_rate_of_change']]
            .tail(30).reset_index(drop=True)  # 30 rows for longest window
        )
        self.high_flow_thresh_ = X['streamflow_today_cumecs'].quantile(0.90)

        # Monthly percentile: median flow per calendar month
        if 'month' in X.columns:
            self.monthly_median_ = (
                X.groupby('month')['streamflow_today_cumecs']
                .median()
                .to_dict()
            )
        else:
            self.monthly_median_ = {}
        return self

    def transform(self, X):
        df = X.copy().reset_index(drop=True)

        n_tail  = len(self.train_tail_)
        tail_df = pd.DataFrame(np.nan, index=range(n_tail), columns=df.columns)
        tail_df['streamflow_today_cumecs'] = self.train_tail_['streamflow_today_cumecs'].values
        tail_df['flow_rate_of_change']     = self.train_tail_['flow_rate_of_change'].values
        combined = pd.concat([tail_df, df], ignore_index=True)

        flow = combined['streamflow_today_cumecs']
        roc  = combined['flow_rate_of_change']

        # ── Lags: short (v2) ──
        for lag in [1, 2, 3]:
            combined[f'flow_lag{lag}'] = flow.shift(lag)

        # ── Lags: long (NEW v3) ──
        for lag in [7, 14, 21]:
            combined[f'flow_lag{lag}'] = flow.shift(lag)

        # ── Rolling: 7d (v2) ──
        combined['rolling_flow_7d'] = flow.rolling(7).mean()
        combined['rolling_std_7d']  = flow.rolling(7).std()
        combined['rolling_max_7d']  = flow.rolling(7).max()

        # ── Rolling: 3d (NEW v3) ──
        combined['rolling_flow_3d'] = flow.rolling(3).mean()
        combined['rolling_std_3d']  = flow.rolling(3).std()
        combined['rolling_max_3d']  = flow.rolling(3).max()

        # ── Rolling: 14d (NEW v3) ──
        combined['rolling_flow_14d'] = flow.rolling(14).mean()
        combined['rolling_std_14d']  = flow.rolling(14).std()

        # ── Rolling: 30d (NEW v3) ──
        combined['rolling_flow_30d'] = flow.rolling(30).mean()
        combined['rolling_std_30d']  = flow.rolling(30).std()

        # ── Acceleration and surge (v2) ──
        combined['flow_acceleration'] = roc - roc.shift(1)
        combined['flow_surge_ratio']  = flow / (combined['rolling_flow_7d'] + 1.0)

        # ── NEW v3: surge ratio over 30d (detects longer-term spike) ──
        combined['flow_surge_ratio_30d'] = flow / (combined['rolling_flow_30d'] + 1.0)

        # ── NEW v3: post-peak decay flag ──
        rolling_max3 = flow.rolling(3).max()
        combined['post_peak_decay'] = ((flow < rolling_max3.shift(1)) & (roc < 0)).astype(np.float32)

        # Drop tail
        df = combined.iloc[n_tail:].reset_index(drop=True)

        # ── High-flow flag (v2) ──
        df['is_high_flow'] = (df['streamflow_today_cumecs'] > self.high_flow_thresh_).astype(np.float32)

        # ── Rain momentum (v2) ──
        if 'antecedent_rain_3d_sum' in df.columns and 'antecedent_rain_7d_sum' in df.columns:
            df['rain_momentum'] = (df['antecedent_rain_3d_sum'] - df['antecedent_rain_7d_sum'] / 2.33).clip(lower=0)

        # ── NEW v3: flow × rain interactions ──
        if 'antecedent_rain_3d_sum' in df.columns:
            df['flow_x_rain3d'] = np.log1p(df['streamflow_today_cumecs'].clip(lower=0)) * np.log1p(df['antecedent_rain_3d_sum'].clip(lower=0))
        if 'antecedent_rain_7d_sum' in df.columns:
            df['flow_x_rain7d'] = np.log1p(df['streamflow_today_cumecs'].clip(lower=0)) * np.log1p(df['antecedent_rain_7d_sum'].clip(lower=0))

        # ── NEW v3: monthly percentile rank ──
        if 'month' in df.columns and self.monthly_median_:
            monthly_med = df['month'].map(self.monthly_median_).fillna(df['streamflow_today_cumecs'].median())
            df['monthly_flow_ratio'] = df['streamflow_today_cumecs'] / (monthly_med + 1.0)

        # ── Log transforms: rain (v2) ──
        for col in self.RAIN_COLS:
            if col in df.columns:
                df[f'log_{col}'] = np.log1p(df[col].clip(lower=0))

        # ── Log transforms: flow lags ──
        for lag in [1, 2, 3, 7, 14, 21]:
            col = f'flow_lag{lag}'
            df[f'log_{col}'] = np.log1p(df[col].clip(lower=0))
        df['log_streamflow_today_cumecs'] = np.log1p(df['streamflow_today_cumecs'].clip(lower=0))

        # ── Log transforms: extra skewed cols (v2) ──
        for col in self.EXTRA_LOG_COLS:
            if col in df.columns:
                df[f'log_{col}'] = np.log1p(df[col].clip(lower=0))

        # ── Cyclical date encoding (v2) ──
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['doy_sin']   = np.sin(2 * np.pi * df['day_of_year'] / 365)
        df['doy_cos']   = np.cos(2 * np.pi * df['day_of_year'] / 365)

        float_cols = df.select_dtypes('float64').columns
        df[float_cols] = df[float_cols].astype(np.float32)
        return df


class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, features):
        self.features = features

    def fit(self, X, y=None):
        self.available_features_ = [f for f in self.features if f in X.columns]
        missing = set(self.features) - set(self.available_features_)
        if missing:
            print(f'[FeatureSelector] {len(missing)} skipped: {sorted(missing)}')
        print(f'[FeatureSelector] Using {len(self.available_features_)} features.')
        return self

    def transform(self, X):
        return X[self.available_features_].values

print('Transformers v3 defined.')

## 4 · Feature List

In [ ]:
ENGINEERED_FEATURES = [
    # Log flow — short lags (v2)
    'log_streamflow_today_cumecs',
    'log_flow_lag1', 'log_flow_lag2', 'log_flow_lag3',
    # Log flow — long lags (NEW v3)
    'log_flow_lag7', 'log_flow_lag14', 'log_flow_lag21',
    # Rolling 7d (v2)
    'rolling_flow_7d', 'rolling_std_7d', 'rolling_max_7d',
    # Rolling 3d (NEW v3)
    'rolling_flow_3d', 'rolling_std_3d', 'rolling_max_3d',
    # Rolling 14d, 30d (NEW v3)
    'rolling_flow_14d', 'rolling_std_14d',
    'rolling_flow_30d', 'rolling_std_30d',
    # Surge and flood signals (v2 + new)
    'flow_acceleration', 'flow_surge_ratio', 'flow_surge_ratio_30d',
    'is_high_flow', 'post_peak_decay',
    # Log rain (v2)
    'log_antecedent_rain_3d_sum', 'log_antecedent_rain_7d_sum',
    'log_antecedent_rain_15d_sum', 'log_antecedent_rain_30d_sum', 'log_antecedent_rain_60d',
    # Rain signals
    'rain_momentum',
    # NEW v3 interaction features
    'flow_x_rain3d', 'flow_x_rain7d', 'monthly_flow_ratio',
    # Log skewed cols (v2)
    'log_soil_saturation_score', 'log_monsoon_cumulative_rain',
    'log_rain_soilmoisture_interaction', 'log_uparea_rain_interaction', 'log_rain_basinsize_interaction',
    # Cyclical
    'month_sin', 'month_cos', 'doy_sin', 'doy_cos',
]

RAW_FEATURES = [
    'soil_saturation_score', 'antecedent_saturation_interaction',
    'streamflow_anomaly_zscore', 'flow_rate_of_change', 'flow_velocity_km_per_day',
    'rainfall_anomaly_zscore', 'upstream_area_scaled', 'dist_to_outlet_scaled',
    'slope_scaled', 'slope_uav_scaled', 'forest_cover_scaled', 'urban_cover_scaled',
    'monsoon_intensity', 'monsoon_cumulative_rain', 'is_post_monsoon_saturated',
    'upstream_rain_mean_scaled', 'upstream_rain_weighted_scaled', 'upstream_rain_lagged_dist_sink',
    'rain_soilmoisture_interaction', 'rain_urban_interaction', 'rain_slope_interaction',
    'rain_basinsize_interaction', 'uparea_rain_interaction',
]

ALL_FEATURES = ENGINEERED_FEATURES + RAW_FEATURES
TARGET       = 'streamflow_tomorrow_cumecs'
LOG_TARGET   = 'log_target'

train[LOG_TARGET] = np.log1p(train[TARGET])

print(f'Total features : {len(ALL_FEATURES)}')

## 5 · Train / Val Split
> **Same as v2:** 80/20 for finding best_iteration only. Full retrain in Section 13.

In [ ]:
DROP_COLS = [TARGET, LOG_TARGET]
split_idx = int(len(train) * 0.80)

X_train_raw = train.iloc[:split_idx].drop(columns=DROP_COLS, errors='ignore')
y_train     = train.iloc[:split_idx][LOG_TARGET]
X_val_raw   = train.iloc[split_idx:].drop(columns=DROP_COLS, errors='ignore')
y_val       = train.iloc[split_idx:][LOG_TARGET]
actual_val  = np.expm1(y_val.values)

print(f'X_train : {X_train_raw.shape}')
print(f'X_val   : {X_val_raw.shape}')

## 6 · Preprocessor (validation phase)

In [ ]:
preprocessor = Pipeline([
    ('feature_eng', FloodFeatureEngineer()),
    ('selector',    FeatureSelector(ALL_FEATURES)),
    ('imputer',     SimpleImputer(strategy='median')),
])

X_train_t = preprocessor.fit_transform(X_train_raw)
X_val_t   = preprocessor.transform(X_val_raw)

print(f'Transformed train : {X_train_t.shape}')
print(f'Transformed val   : {X_val_t.shape}')

## 7 · XGBoost

In [ ]:
xgb_model = XGBRegressor(
    n_estimators          = 4000,
    learning_rate         = 0.015,   # slower = more trees = better
    max_depth             = 7,
    subsample             = 0.8,
    colsample_bytree      = 0.65,
    min_child_weight      = 10,
    gamma                 = 0.1,
    reg_alpha             = 0.1,
    reg_lambda            = 2.0,
    early_stopping_rounds = 100,
    eval_metric           = 'mae',
    random_state          = RANDOM_STATE,
    n_jobs                = -1,
    tree_method           = 'hist',
)

xgb_model.fit(
    X_train_t, y_train,
    eval_set  = [(X_val_t, y_val)],
    verbose   = 500,
)

xgb_preds = np.expm1(xgb_model.predict(X_val_t))
print(f'\nXGB best_iteration : {xgb_model.best_iteration}')
print(f'XGBoost  KGE : {compute_kge(xgb_preds, actual_val):.4f}')
print(f'XGBoost  NSE : {compute_nse(xgb_preds, actual_val):.4f}')

## 8 · LightGBM

In [ ]:
lgb_model = lgb.LGBMRegressor(
    n_estimators        = 4000,
    learning_rate       = 0.015,
    num_leaves          = 127,
    subsample           = 0.8,
    colsample_bytree    = 0.65,
    min_child_samples   = 15,
    reg_alpha           = 0.1,
    reg_lambda          = 2.0,
    random_state        = RANDOM_STATE,
    n_jobs              = -1,
    verbose             = -1,
)

lgb_model.fit(
    X_train_t, y_train,
    eval_set  = [(X_val_t, y_val)],
    callbacks = [
        lgb.early_stopping(100, verbose=False),
        lgb.log_evaluation(500),
    ],
)

lgb_preds = np.expm1(lgb_model.predict(X_val_t))
print(f'\nLGB best_iteration : {lgb_model.best_iteration_}')
print(f'LightGBM  KGE : {compute_kge(lgb_preds, actual_val):.4f}')
print(f'LightGBM  NSE : {compute_nse(lgb_preds, actual_val):.4f}')

## 9 · CatBoost

In [ ]:
cat_model = CatBoostRegressor(
    iterations            = 4000,
    learning_rate         = 0.015,
    depth                 = 8,
    subsample             = 0.8,
    colsample_bylevel     = 0.65,
    l2_leaf_reg           = 2.0,
    early_stopping_rounds = 100,
    eval_metric           = 'MAE',
    random_seed           = RANDOM_STATE,
    verbose               = 500,
)

cat_model.fit(
    X_train_t, y_train,
    eval_set  = (X_val_t, y_val),
)

cat_preds = np.expm1(cat_model.predict(X_val_t))
print(f'\nCatBoost best_iteration : {cat_model.best_iteration_}')
print(f'CatBoost  KGE : {compute_kge(cat_preds, actual_val):.4f}')
print(f'CatBoost  NSE : {compute_nse(cat_preds, actual_val):.4f}')

## 10 · Optimised Blend — scipy minimize on KGE
> Better than grid search: scipy finds the exact optimal weights in continuous space.

In [ ]:
xgb_log_val = xgb_model.predict(X_val_t)
lgb_log_val = lgb_model.predict(X_val_t)
cat_log_val = cat_model.predict(X_val_t)

# ── scipy optimize (continuous, unconstrained → project to simplex) ──
def neg_kge(weights):
    w = np.array(weights)
    w = np.clip(w, 0, 1)
    w = w / w.sum()
    blend = np.expm1(w[0]*xgb_log_val + w[1]*lgb_log_val + w[2]*cat_log_val)
    return -compute_kge(blend, actual_val)

result = minimize(
    neg_kge,
    x0        = [0.33, 0.33, 0.34],
    method    = 'Nelder-Mead',
    options   = {'maxiter': 5000, 'xatol': 1e-6, 'fatol': 1e-8},
)

raw_w        = np.clip(result.x, 0, 1)
opt_weights  = raw_w / raw_w.sum()
best_w1, best_w2, best_w3 = opt_weights

# Also run fine grid to avoid local minimum
grid_kge, grid_w1, grid_w2 = -np.inf, 0.33, 0.33
for w1 in np.arange(0.0, 1.01, 0.02):
    for w2 in np.arange(0.0, 1.01 - w1, 0.02):
        w3    = 1.0 - w1 - w2
        blend = np.expm1(w1*xgb_log_val + w2*lgb_log_val + w3*cat_log_val)
        kge   = compute_kge(blend, actual_val)
        if kge > grid_kge:
            grid_kge, grid_w1, grid_w2 = kge, w1, w2
grid_w3 = round(1.0 - grid_w1 - grid_w2, 4)

# Use whichever is better
scipy_kge = -result.fun
if scipy_kge >= grid_kge:
    print(f'scipy wins  →  XGB={best_w1:.3f}  LGB={best_w2:.3f}  CAT={best_w3:.3f}  KGE={scipy_kge:.4f}')
else:
    best_w1, best_w2, best_w3 = grid_w1, grid_w2, grid_w3
    print(f'grid wins   →  XGB={best_w1:.3f}  LGB={best_w2:.3f}  CAT={best_w3:.3f}  KGE={grid_kge:.4f}')

ensemble_val = np.expm1(best_w1*xgb_log_val + best_w2*lgb_log_val + best_w3*cat_log_val)
best_kge     = compute_kge(ensemble_val, actual_val)
print(f'\nEnsemble val KGE : {best_kge:.4f}')
print(f'Ensemble val NSE : {compute_nse(ensemble_val, actual_val):.4f}')

## 11 · Ridge Meta-Learner Stacking
> Train a Ridge regression on top of XGB+LGB+CAT predictions.
> Learns residual corrections that no single model captures.

In [ ]:
# Stack in log space — all three base learner predictions as features
meta_train = np.column_stack([xgb_log_val, lgb_log_val, cat_log_val])

# Also add the raw flow feature (today's flow) — persistence signal
if 'log_streamflow_today_cumecs' in preprocessor.named_steps['selector'].available_features_:
    feat_idx = preprocessor.named_steps['selector'].available_features_.index('log_streamflow_today_cumecs')
    meta_train = np.column_stack([meta_train, X_val_t[:, feat_idx]])

# Fit Ridge on validation predictions vs actual (log scale)
ridge = Ridge(alpha=10.0)
ridge.fit(meta_train, y_val.values)

stacked_log_val = ridge.predict(meta_train)
stacked_val     = np.expm1(stacked_log_val)
stacked_kge     = compute_kge(stacked_val, actual_val)

print(f'Ridge stacked KGE : {stacked_kge:.4f}')
print(f'Ridge stacked NSE : {compute_nse(stacked_val, actual_val):.4f}')
print()

# Use stacked if better, else ensemble
if stacked_kge > best_kge:
    print(f'Stacking wins! ({stacked_kge:.4f} > {best_kge:.4f}) — will use Ridge for submission')
    USE_STACKING = True
else:
    print(f'Ensemble blend wins ({best_kge:.4f} >= {stacked_kge:.4f}) — will use blend for submission')
    USE_STACKING = False

## 12 · Regime Analysis + Diagnostics

In [ ]:
final_val = stacked_val if USE_STACKING else ensemble_val
final_kge = stacked_kge if USE_STACKING else best_kge

regime_analysis(final_val,  actual_val, label='FINAL (best method)')
regime_analysis(xgb_preds,  actual_val, label='XGBoost')
regime_analysis(lgb_preds,  actual_val, label='LightGBM')
regime_analysis(cat_preds,  actual_val, label='CatBoost')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].scatter(actual_val, final_val, alpha=0.15, s=2, color='steelblue')
lim = max(actual_val.max(), final_val.max())
axes[0].plot([0, lim], [0, lim], 'r--', lw=1)
axes[0].set_xlabel('Actual (cumecs)')
axes[0].set_ylabel('Predicted (cumecs)')
axes[0].set_title(f'Actual vs Predicted  KGE={final_kge:.4f}')

residuals = actual_val - final_val
axes[1].hist(residuals, bins=100, color='coral', edgecolor='none')
axes[1].axvline(0, color='black', lw=1)
axes[1].set_xlabel('Residual (cumecs)')
axes[1].set_title(f'Residuals  MAE={mean_absolute_error(actual_val, final_val):.2f}')

n = 500
axes[2].plot(actual_val[-n:], color='black',     lw=1.0, label='Actual')
axes[2].plot(final_val[-n:],  color='steelblue', lw=0.8, alpha=0.85, label='Predicted')
axes[2].set_title('Last 500 val rows')
axes[2].legend(fontsize=8)
axes[2].set_xlabel('Day')
axes[2].set_ylabel('Streamflow (cumecs)')

plt.tight_layout()
plt.savefig('validation_diagnostics_v3.png', dpi=120)
plt.show()

In [ ]:
feat_names = preprocessor.named_steps['selector'].available_features_
xgb_imp    = xgb_model.feature_importances_
lgb_imp    = lgb_model.feature_importances_ / lgb_model.feature_importances_.sum()
avg_imp    = (xgb_imp / xgb_imp.sum() + lgb_imp) / 2

imp_df = (
    pd.DataFrame({'feature': feat_names, 'importance': avg_imp})
    .sort_values('importance', ascending=False)
    .head(30)
)

plt.figure(figsize=(10, 10))
plt.barh(imp_df['feature'][::-1], imp_df['importance'][::-1], color='steelblue')
plt.xlabel('Avg Normalised Importance (XGB + LGB)')
plt.title('Top-30 Features — v3')
plt.tight_layout()
plt.savefig('feature_importance_v3.png', dpi=120)
plt.show()
print('Top 10:')
print(imp_df[['feature','importance']].head(10).to_string(index=False))

## 13 · ⚡ FULL RETRAIN ON 100% DATA
> **Do not skip. Do not submit predictions from the 80%-trained models above.**

In [ ]:
print('=== FULL RETRAIN ===')
print(f'Rows     : {len(train):,}')
print(f'XGB iter : {xgb_model.best_iteration}')
print(f'LGB iter : {lgb_model.best_iteration_}')
print(f'CAT iter : {cat_model.best_iteration_}')
print(f'Blend    : XGB={best_w1:.3f}  LGB={best_w2:.3f}  CAT={best_w3:.3f}')
print(f'Stacking : {USE_STACKING}')

X_full = train.drop(columns=DROP_COLS, errors='ignore')
y_full = train[LOG_TARGET]

preprocessor_full = Pipeline([
    ('feature_eng', FloodFeatureEngineer()),
    ('selector',    FeatureSelector(ALL_FEATURES)),
    ('imputer',     SimpleImputer(strategy='median')),
])
X_full_t = preprocessor_full.fit_transform(X_full)
print(f'Full set transformed : {X_full_t.shape}')

In [ ]:
final_xgb = XGBRegressor(
    n_estimators     = xgb_model.best_iteration,
    learning_rate    = 0.015,
    max_depth        = 7,
    subsample        = 0.8,
    colsample_bytree = 0.65,
    min_child_weight = 10,
    gamma            = 0.1,
    reg_alpha        = 0.1,
    reg_lambda       = 2.0,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    tree_method      = 'hist',
)
final_xgb.fit(X_full_t, y_full, verbose=False)
print(f'✓ Final XGBoost  ({xgb_model.best_iteration} trees on {len(train):,} rows)')

In [ ]:
final_lgb = lgb.LGBMRegressor(
    n_estimators      = lgb_model.best_iteration_,
    learning_rate     = 0.015,
    num_leaves        = 127,
    subsample         = 0.8,
    colsample_bytree  = 0.65,
    min_child_samples = 15,
    reg_alpha         = 0.1,
    reg_lambda        = 2.0,
    random_state      = RANDOM_STATE,
    n_jobs            = -1,
    verbose           = -1,
)
final_lgb.fit(X_full_t, y_full)
print(f'✓ Final LightGBM ({lgb_model.best_iteration_} trees on {len(train):,} rows)')

In [ ]:
final_cat = CatBoostRegressor(
    iterations        = cat_model.best_iteration_,
    learning_rate     = 0.015,
    depth             = 8,
    subsample         = 0.8,
    colsample_bylevel = 0.65,
    l2_leaf_reg       = 2.0,
    random_seed       = RANDOM_STATE,
    verbose           = 0,
)
final_cat.fit(X_full_t, y_full)
print(f'✓ Final CatBoost ({cat_model.best_iteration_} trees on {len(train):,} rows)')

## 14 · Generate Submission

In [ ]:
X_test_t = preprocessor_full.transform(test)
print(f'Test transformed : {X_test_t.shape}')

xgb_log_test = final_xgb.predict(X_test_t)
lgb_log_test = final_lgb.predict(X_test_t)
cat_log_test = final_cat.predict(X_test_t)

if USE_STACKING:
    # Build meta features for test (same columns as meta_train)
    meta_test = np.column_stack([xgb_log_test, lgb_log_test, cat_log_test])
    if 'log_streamflow_today_cumecs' in preprocessor_full.named_steps['selector'].available_features_:
        feat_idx = preprocessor_full.named_steps['selector'].available_features_.index('log_streamflow_today_cumecs')
        meta_test = np.column_stack([meta_test, X_test_t[:, feat_idx]])
    test_preds = np.expm1(ridge.predict(meta_test))
    print('Using Ridge stacked predictions')
else:
    test_preds = np.expm1(
        best_w1 * xgb_log_test +
        best_w2 * lgb_log_test +
        best_w3 * cat_log_test
    )
    print('Using weighted ensemble predictions')

test_preds = np.clip(test_preds, 0, None)

submission = sub[['row_id']].copy()
submission['streamflow_tomorrow_cumecs'] = test_preds
submission.to_csv('submission_v3_rank1.csv', index=False)

print(f'\n✓ Saved submission_v3_rank1.csv  —  {len(submission):,} rows')
print(f'Range : {test_preds.min():.2f} – {test_preds.max():.2f} cumecs')
print(f'Mean  : {test_preds.mean():.2f}  |  Std : {test_preds.std():.2f}')
submission.head()

## 15 · Sanity Check

In [ ]:
assert len(submission) == len(sub),                                 'Row mismatch!'
assert submission.isnull().sum().sum() == 0,                        'NaNs found!'
assert (submission['streamflow_tomorrow_cumecs'] >= 0).all(),       'Negatives!'
assert (submission['row_id'].values == sub['row_id'].values).all(), 'row_id mismatch!'

print('✓ Rows     :', len(submission))
print('✓ No nulls')
print('✓ No negatives')
print('✓ row_id match')
print()
print(submission['streamflow_tomorrow_cumecs'].describe().round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(test_preds, bins=100, color='steelblue', edgecolor='none')
axes[0].set_xlabel('Predicted (cumecs)')
axes[0].set_title('Test Prediction Distribution')

axes[1].hist(train[TARGET], bins=100, color='coral',     alpha=0.6, edgecolor='none', density=True, label='Train actual')
axes[1].hist(test_preds,    bins=100, color='steelblue', alpha=0.6, edgecolor='none', density=True, label='Test predicted')
axes[1].legend()
axes[1].set_title('Train vs Test distribution')

plt.tight_layout()
plt.savefig('submission_sanity_v3.png', dpi=120)
plt.show()
print('\n✓ All checks passed.')
print('Submit submission_v3_rank1.csv to Kaggle NOW.')